In [1]:
import os
import pickle

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, balanced_accuracy_score, recall_score
from sklearn.neural_network import MLPClassifier

from xgboost import XGBClassifier

from CSDTools.features import FEATURIZERS, get_or_build_features

In [2]:
CSV_FILE = 'csd_ml.csv'
data = pd.read_csv(CSV_FILE)
data


,smiles_canonical,is_polymorphic,num_inchi_groups,max_distinct_forms,min_distinct_forms
0,B(N1CCc2ccccc21)N1CCc2ccccc21,0,1,1,1
1,B(Nc1ccccc1-c1ccccc1)N(BNc1ccccc1-c1ccccc1)c1c...,0,1,1,1
2,B1C=CC=C(c2ccccn2)N1,0,1,1,1
3,B1N(CP(C2CCCCC2)C2CCCCC2)c2ccccc2N1CP(C1CCCCC1...,0,1,1,1
4,B1N(CP(c2ccccc2)c2ccccc2)c2ccccc2N1CP(c1ccccc1...,0,1,1,1
...,...,...,...,...,...
292368,n1onc2c1NC1Nc3nonc3NC1N2,0,1,1,1
292369,n1onc2c1Nc1nonc1-c1nonc1-2,0,1,1,1
292370,n1snc2c1SC(=C1Sc3nsnc3S1)S2,0,1,1,1
292371,s1c2sc3sc4sc5sc6sc7sc8sc1c1c2c3c4c5c6c7c81,1,1,2,2


In [3]:
smiles_list  = data['smiles_canonical'].to_list()
labels_full  = data['is_polymorphic'].to_list()


In [4]:
from sklearn.ensemble import (
    RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier,
    AdaBoostClassifier
)
from sklearn.linear_model import LogisticRegression, SGDClassifier, RidgeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from imblearn.ensemble import (
    BalancedRandomForestClassifier, EasyEnsembleClassifier, BalancedBaggingClassifier
)

RESULTS_CACHE   = "training_results.csv"
MODEL_CACHE_DIR = "model_cache"
os.makedirs(MODEL_CACHE_DIR, exist_ok=True)

MODELS = {
    "XGBClassifier": XGBClassifier,
    "MLPClassifier": MLPClassifier,
    "LGBMClassifier": LGBMClassifier,
    "HistGB": HistGradientBoostingClassifier,
    "RandomForest": RandomForestClassifier,
    "ExtraTrees": ExtraTreesClassifier,
    "BalancedRF": BalancedRandomForestClassifier,
    "LogisticRegression": LogisticRegression,
    "CatBoost": CatBoostClassifier,
    "AdaBoost": AdaBoostClassifier,
    "EasyEnsemble": EasyEnsembleClassifier,
    "SGD": SGDClassifier,
    "Ridge": RidgeClassifier,
    "GaussianNB": GaussianNB,
    "KNN": KNeighborsClassifier,
    "BalancedBagging": BalancedBaggingClassifier,
}

# Modèles qui acceptent scale_pos_weight à la construction (comme XGB)
SCALE_POS_WEIGHT_MODELS = {"XGBClassifier", "LGBMClassifier"}

# Modèles qui acceptent class_weight='balanced' à la construction
CLASS_WEIGHT_MODELS = {"RandomForest", "ExtraTrees", "HistGB", "LogisticRegression", "SGD", "Ridge"}

# Modèles qui acceptent n_jobs (à combiner avec class_weight quand les deux s'appliquent)
N_JOBS_MODELS = {"RandomForest", "ExtraTrees", "BalancedRF", "SGD", "KNN", "EasyEnsemble", "BalancedBagging"}

# Le reste (MLP, CatBoost, AdaBoost, GaussianNB, KNN seul) gère le déséquilibre autrement ou pas du tout



if os.path.exists(RESULTS_CACHE):
    results = pd.read_csv(RESULTS_CACHE).to_dict("records")
else:
    results = []
done_results = {(r["featurizer"], r["model"]): r for r in results}

for entry in FEATURIZERS:
    cache_key, featurizer, kwargs = entry if isinstance(entry, tuple) else (entry, entry, {})

    pending_models = []
    for model_name in MODELS:
        if (cache_key, model_name) in done_results:
            r = done_results[(cache_key, model_name)]
            print(f"⏭ {cache_key:20s} / {model_name:15s} — "
                  f"acc={r['accuracy']:.3f}  bal_acc={r['balanced_accuracy']:.3f}  recall={r['recall']:.3f}  (cache)")
        else:
            pending_models.append(model_name)

    if not pending_models:
        continue

    try:
        x, valid_ids = get_or_build_features(CSV_FILE, featurizer, smiles_list=smiles_list,
                                              cache_key=cache_key, **kwargs)
    except Exception as e:
        print(f"⛔ {cache_key} featurization failed: {e}")
        continue

    label_list = [labels_full[i] for i in valid_ids]
    x_train, x_test, y_train, y_test = train_test_split(
        x, label_list, test_size=0.2, random_state=42, stratify=label_list
    )

    n_neg = sum(1 for y in y_train if y == 0)
    n_pos = sum(1 for y in y_train if y == 1)
    scale = n_neg / n_pos if n_pos else 1.0
    sample_weight = np.array([scale if y == 1 else 1.0 for y in y_train])

    for model_name in pending_models:
        model_cls = MODELS[model_name]
        try:
            if model_name in SCALE_POS_WEIGHT_MODELS:
                model = model_cls(scale_pos_weight=scale)
                model.fit(x_train, y_train)
            elif model_name in N_JOBS_MODELS and model_name in CLASS_WEIGHT_MODELS:
                if model_name == "RandomForest":
                    model = model_cls(class_weight="balanced_subsample", n_jobs=-1)
                else:
                    model = model_cls(class_weight="balanced", n_jobs=-1)
                model.fit(x_train, y_train)
            elif model_name in N_JOBS_MODELS:  # BalancedRF, pas de class_weight
                model = model_cls(n_jobs=-1)
                model.fit(x_train, y_train)
            elif model_name in CLASS_WEIGHT_MODELS:
                model = model_cls(class_weight="balanced")
                model.fit(x_train, y_train)
            elif model_name == "MLPClassifier":
                model = model_cls(max_iter=300, early_stopping=True)
                model.fit(x_train, y_train, sample_weight=sample_weight)
            elif model_name == "CatBoost":
                model = model_cls(class_weights=[1, scale], verbose=False, iterations=500)
                model.fit(x_train, y_train)
            elif model_name == "AdaBoost":
                model = model_cls(
                    estimator=DecisionTreeClassifier(class_weight="balanced", max_depth=5),
                    n_estimators=100
                )
                model.fit(x_train, y_train)
            else:
                model = model_cls()
                model.fit(x_train, y_train)

            y_pred = model.predict(x_test)
            result = {
                "featurizer":        cache_key,
                "model":             model_name,
                "n_features":        x.shape[1],
                "accuracy":          accuracy_score(y_test, y_pred),
                "balanced_accuracy": balanced_accuracy_score(y_test, y_pred),
                "recall":            recall_score(y_test, y_pred),
            }
            results.append(result)
            done_results[(cache_key, model_name)] = result
            pd.DataFrame(results).to_csv(RESULTS_CACHE, index=False)

            with open(os.path.join(MODEL_CACHE_DIR, f"{cache_key}__{model_name}.pkl"), "wb") as f:
                pickle.dump(model, f)

            print(f"✅ {cache_key:20s} / {model_name:15s} — "
                  f"acc={result['accuracy']:.3f}  bal_acc={result['balanced_accuracy']:.3f}  recall={result['recall']:.3f}")
        except Exception as e:
            print(f"⛔ {cache_key}/{model_name} failed: {e}")

⏭ avalon               / XGBClassifier   — acc=0.860  bal_acc=0.680  recall=0.492  (cache)
⏭ avalon               / MLPClassifier   — acc=0.801  bal_acc=0.684  recall=0.562  (cache)
⏭ avalon               / LGBMClassifier  — acc=0.755  bal_acc=0.684  recall=0.611  (cache)
⏭ avalon               / HistGB          — acc=0.739  bal_acc=0.693  recall=0.644  (cache)
⏭ avalon               / RandomForest    — acc=0.980  bal_acc=0.514  recall=0.029  (cache)
⏭ avalon               / ExtraTrees      — acc=0.978  bal_acc=0.523  recall=0.051  (cache)
⏭ avalon               / BalancedRF      — acc=0.849  bal_acc=0.680  recall=0.504  (cache)
⏭ avalon               / LogisticRegression — acc=0.680  bal_acc=0.680  recall=0.679  (cache)
⏭ avalon               / CatBoost        — acc=0.869  bal_acc=0.674  recall=0.471  (cache)
⏭ avalon               / AdaBoost        — acc=0.697  bal_acc=0.621  recall=0.542  (cache)
⏭ avalon               / EasyEnsemble    — acc=0.673  bal_acc=0.682  recall=0.691  (cac

KeyboardInterrupt: 

In [5]:
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

results_df = pd.DataFrame(results).sort_values("balanced_accuracy", ascending=False).reset_index(drop=True)
results_df

,featurizer,model,n_features,accuracy,balanced_accuracy,recall
0,desc2D,HistGB,223,0.754536,0.709938,0.663611
1,ecfp,MLPClassifier,2048,0.798136,0.704728,0.607699
2,atompair-count,HistGB,2048,0.748354,0.704091,0.658112
3,topological-count,BalancedRF,2048,0.786182,0.703583,0.617782
4,fcfp,MLPClassifier,2048,0.774177,0.702412,0.627864
5,secfp,GaussianNB,2048,0.703326,0.701380,0.699358
6,fcfp,LGBMClassifier,2048,0.734912,0.701288,0.666361
7,ecfp,LGBMClassifier,2048,0.734912,0.701288,0.666361
8,topological,MLPClassifier,2048,0.742232,0.697824,0.651696
9,ecfp-count,BalancedRF,2048,0.814793,0.697479,0.575619


In [6]:
best_featurizer, best_model_name = results_df.loc[0, ["featurizer", "model"]]
with open(os.path.join(MODEL_CACHE_DIR, f"{best_featurizer}__{best_model_name}.pkl"), "rb") as f:
    best_model = pickle.load(f)

with open("best_polymorph_model.pkl", "wb") as f:
    pickle.dump(best_model, f)

print(f"Meilleur modèle sauvegardé : {best_model_name} / {best_featurizer}")

Meilleur modèle sauvegardé : HistGB / desc2D


In [4]:
import time
import json as jsonlib

from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from scipy.stats import randint, uniform, loguniform

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression, SGDClassifier, RidgeClassifier
from sklearn.naive_bayes import GaussianNB
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from imblearn.ensemble import BalancedRandomForestClassifier

RESULTS_CACHE   = "training_results.csv"

TOP_N = 50
TUNING_RESULTS_CACHE  = "tuning_results.csv"
TUNED_MODEL_CACHE_DIR = "tuned_model_cache"
os.makedirs(TUNED_MODEL_CACHE_DIR, exist_ok=True)

base_results = pd.read_csv(RESULTS_CACHE)
before_lookup = {(r["featurizer"], r["model"]): r for r in base_results.to_dict("records")}

top_combos = (
    base_results
    .sort_values("balanced_accuracy", ascending=False)
    .head(TOP_N)[["featurizer", "model"]]
    .itertuples(index=False, name=None)
)
top_combos = list(top_combos)

# Rapides d'abord, lents en dernier (ordre observé empiriquement sur les runs déjà faits)
MODEL_SPEED_ORDER = [
    "Ridge", "GaussianNB", "SGD", "LGBMClassifier", "LogisticRegression",
    "BalancedRF", "XGBClassifier", "CatBoost", "HistGB", "MLPClassifier",
]
top_combos = sorted(top_combos, key=lambda c: MODEL_SPEED_ORDER.index(c[1]))

FEATURIZER_LOOKUP = {}
for entry in FEATURIZERS:
    cache_key, featurizer, kwargs = entry if isinstance(entry, tuple) else (entry, entry, {})
    FEATURIZER_LOOKUP[cache_key] = (featurizer, kwargs)

PARAM_DISTRIBUTIONS = {
    "HistGB": dict(
        max_iter=randint(100, 1000),
        max_depth=randint(3, 15),
        max_leaf_nodes=randint(15, 127),
        learning_rate=loguniform(0.01, 0.3),
        l2_regularization=loguniform(1e-3, 10),
        min_samples_leaf=randint(5, 100),
    ),
    "LGBMClassifier": dict(
        n_estimators=randint(100, 600),
        num_leaves=randint(15, 200),
        max_depth=randint(3, 15),
        learning_rate=loguniform(0.01, 0.3),
        min_child_samples=randint(5, 200),
        reg_alpha=loguniform(1e-3, 10),
        reg_lambda=loguniform(1e-3, 10),
        subsample=uniform(0.6, 0.4),
        colsample_bytree=uniform(0.6, 0.4),
    ),
    "XGBClassifier": dict(
        n_estimators=randint(100, 1000),
        max_depth=randint(3, 12),
        learning_rate=loguniform(0.01, 0.3),
        min_child_weight=randint(1, 20),
        subsample=uniform(0.6, 0.4),
        colsample_bytree=uniform(0.6, 0.4),
        reg_alpha=loguniform(1e-3, 30),
        reg_lambda=loguniform(1e-3, 10),
        gamma=uniform(0, 10),
    ),
    "CatBoost": dict(
        iterations=randint(200, 800),
        depth=randint(4, 10),
        learning_rate=loguniform(0.01, 0.3),
        l2_leaf_reg=loguniform(1, 20),
    ),
    "BalancedRF": dict(
        n_estimators=randint(100, 500),
        max_depth=randint(5, 30),
        min_samples_leaf=randint(1, 50),
        max_features=uniform(0.1, 0.9),
    ),
    "MLPClassifier": dict(
        hidden_layer_sizes=[(64,), (128,), (256,), (128, 64), (256, 128), (128, 64, 32)],
        alpha=loguniform(1e-5, 1e-1),
        learning_rate_init=loguniform(1e-4, 1e-2),
        activation=["relu", "tanh"],
    ),
    "Ridge": dict(
        alpha=loguniform(1e-3, 300),
    ),
    "GaussianNB": dict(
        var_smoothing=loguniform(1e-11, 1e-6),
    ),
    "LogisticRegression": dict(
        C=loguniform(1e-4, 100),
    ),
    "SGD": dict(
        alpha=loguniform(1e-6, 1e-1),
        loss=["hinge", "log_loss", "modified_huber"],
        penalty=["l2", "l1", "elasticnet"],
    ),
}

# Modèles lents / ensembles coûteux -> moins de candidats pour rester raisonnable
N_ITER_BY_MODEL = {
    "MLPClassifier": 10,
    "CatBoost": 12,
    "BalancedRF": 12,
    "Ridge": 8,
    "GaussianNB": 8,
    "SGD": 8,
}
DEFAULT_N_ITER = 20
CV_FOLDS = 3


def build_base_estimator(model_name, scale):
    if model_name == "XGBClassifier":
        return XGBClassifier(scale_pos_weight=scale)
    if model_name == "LGBMClassifier":
        return LGBMClassifier(scale_pos_weight=scale, verbosity=-1)
    if model_name == "CatBoost":
        # scale_pos_weight (float) plutôt que class_weights (liste) : cette dernière
        # cassait le clone() de sklearn utilisé par RandomizedSearchCV
        return CatBoostClassifier(scale_pos_weight=scale, verbose=False)
    if model_name == "HistGB":
        return HistGradientBoostingClassifier(class_weight="balanced")
    if model_name == "BalancedRF":
        return BalancedRandomForestClassifier(n_jobs=-1)
    if model_name == "Ridge":
        return RidgeClassifier(class_weight="balanced")
    if model_name == "GaussianNB":
        return GaussianNB()
    if model_name == "MLPClassifier":
        return MLPClassifier(max_iter=300, early_stopping=True)
    if model_name == "LogisticRegression":
        return LogisticRegression(class_weight="balanced")
    if model_name == "SGD":
        # max_iter plafonné + early_stopping : évite la convergence très lente
        # observée avec certains tirages (alpha faible / elasticnet) sur ce dataset
        return SGDClassifier(class_weight="balanced", n_jobs=-1, max_iter=200, early_stopping=True)
    raise ValueError(f"Pas de config de tuning pour {model_name}")


def tune_combo(model_name, x_train, y_train, sample_weight, scale, n_iter, cv):
    base = build_base_estimator(model_name, scale)
    search = RandomizedSearchCV(
        base, PARAM_DISTRIBUTIONS[model_name],
        n_iter=n_iter,
        cv=StratifiedKFold(cv, shuffle=True, random_state=42),
        scoring="balanced_accuracy",
        random_state=42,
        n_jobs=1,
        refit=True,
    )
    fit_kwargs = {}
    if model_name == "MLPClassifier":
        fit_kwargs["sample_weight"] = sample_weight
    search.fit(x_train, y_train, **fit_kwargs)
    return search


In [8]:
# Test rapide sur 1 combo (n_iter/cv réduits) pour valider que le pipeline tourne avant le run complet
_test_key, _test_model = "secfp", "Ridge"
_featurizer, _kwargs = FEATURIZER_LOOKUP[_test_key]

_x, _valid_ids = get_or_build_features(CSV_FILE, _featurizer, smiles_list=smiles_list,
                                        cache_key=_test_key, **_kwargs)
_label_list = [labels_full[i] for i in _valid_ids]
_x_train, _x_test, _y_train, _y_test = train_test_split(
    _x, _label_list, test_size=0.2, random_state=42, stratify=_label_list
)
_n_neg = sum(1 for y in _y_train if y == 0)
_n_pos = sum(1 for y in _y_train if y == 1)
_scale = _n_neg / _n_pos if _n_pos else 1.0
_sample_weight = np.array([_scale if y == 1 else 1.0 for y in _y_train])

_t0 = time.time()
_search = tune_combo(_test_model, _x_train, _y_train, _sample_weight, _scale, n_iter=3, cv=2)
_elapsed = time.time() - _t0

_y_pred = _search.best_estimator_.predict(_x_test)
print(f"OK — {_test_key}/{_test_model} tuning (3 iters, cv=2) en {_elapsed:.1f}s")
_bal_acc_before = before_lookup[(_test_key, _test_model)]["balanced_accuracy"]
print(f"balanced_accuracy avant tuning : {_bal_acc_before:.3f}")
print(f"balanced_accuracy après ce mini-test : {balanced_accuracy_score(_y_test, _y_pred):.3f}")
print(f"best_params (mini-test) : {_search.best_params_}")


Cache chargé [secfp / csd_ml.csv] : (292373, 2048)
OK — secfp/Ridge tuning (3 iters, cv=2) en 39.4s
balanced_accuracy avant tuning : 0.695
balanced_accuracy après ce mini-test : 0.695
best_params (mini-test) : {'alpha': np.float64(56.69849511478853)}


In [ ]:
if os.path.exists(TUNING_RESULTS_CACHE):
    tuning_results = pd.read_csv(TUNING_RESULTS_CACHE).to_dict("records")
else:
    tuning_results = []
tuning_done = {(r["featurizer"], r["model"]): r for r in tuning_results}

for cache_key, model_name in top_combos:
    if (cache_key, model_name) in tuning_done:
        r = tuning_done[(cache_key, model_name)]
        print(f"⏭ {cache_key:20s} / {model_name:15s} — "
              f"bal_acc={r['balanced_accuracy']:.3f} (avant tuning: {r['balanced_accuracy_before']:.3f})  (cache)")
        continue

    featurizer, kwargs = FEATURIZER_LOOKUP[cache_key]
    try:
        x, valid_ids = get_or_build_features(CSV_FILE, featurizer, smiles_list=smiles_list,
                                              cache_key=cache_key, **kwargs)
    except Exception as e:
        print(f"⛔ {cache_key} featurization failed: {e}")
        continue

    label_list = [labels_full[i] for i in valid_ids]
    x_train, x_test, y_train, y_test = train_test_split(
        x, label_list, test_size=0.2, random_state=42, stratify=label_list
    )

    n_neg = sum(1 for y in y_train if y == 0)
    n_pos = sum(1 for y in y_train if y == 1)
    scale = n_neg / n_pos if n_pos else 1.0
    sample_weight = np.array([scale if y == 1 else 1.0 for y in y_train])

    n_iter = N_ITER_BY_MODEL.get(model_name, DEFAULT_N_ITER)

    try:
        t0 = time.time()
        search = tune_combo(model_name, x_train, y_train, sample_weight, scale, n_iter, CV_FOLDS)
        elapsed = time.time() - t0

        best_model = search.best_estimator_
        y_pred = best_model.predict(x_test)
        bal_acc_before = before_lookup.get((cache_key, model_name), {}).get("balanced_accuracy")

        result = {
            "featurizer":               cache_key,
            "model":                    model_name,
            "balanced_accuracy_before": bal_acc_before,
            "accuracy":                 accuracy_score(y_test, y_pred),
            "balanced_accuracy":        balanced_accuracy_score(y_test, y_pred),
            "recall":                   recall_score(y_test, y_pred),
            "cv_best_score":            search.best_score_,
            "best_params":              jsonlib.dumps(search.best_params_),
            "n_iter":                   n_iter,
            "elapsed_sec":              round(elapsed, 1),
        }
        tuning_results.append(result)
        tuning_done[(cache_key, model_name)] = result
        pd.DataFrame(tuning_results).to_csv(TUNING_RESULTS_CACHE, index=False)

        with open(os.path.join(TUNED_MODEL_CACHE_DIR, f"{cache_key}__{model_name}.pkl"), "wb") as f:
            pickle.dump(best_model, f)

        delta = result["balanced_accuracy"] - bal_acc_before if bal_acc_before is not None else float("nan")
        print(f"✅ {cache_key:20s} / {model_name:15s} — "
              f"bal_acc={result['balanced_accuracy']:.3f} (avant: {bal_acc_before:.3f}, Δ={delta:+.3f})  "
              f"[{elapsed:.0f}s, {n_iter} iters]")
    except Exception as e:
        print(f"⛔ {cache_key}/{model_name} failed: {e}")


Cache chargé [secfp / csd_ml.csv] : (292373, 2048)
✅ secfp                / Ridge           — bal_acc=0.696 (avant: 0.695, Δ=+0.001)  [167s, 8 iters]
Cache chargé [ecfp / csd_ml.csv] : (292373, 2048)
✅ ecfp                 / Ridge           — bal_acc=0.696 (avant: 0.688, Δ=+0.007)  [167s, 8 iters]
Cache chargé [fcfp / csd_ml.csv] : (292373, 2048)
✅ fcfp                 / Ridge           — bal_acc=0.696 (avant: 0.688, Δ=+0.007)  [177s, 8 iters]
⏭ secfp                / GaussianNB      — bal_acc=0.701 (avant tuning: 0.701)  (cache)
⏭ secfp                / SGD             — bal_acc=0.707 (avant tuning: 0.687)  (cache)
Cache chargé [fcfp / csd_ml.csv] : (292373, 2048)


/home/arthur/miniforge3/envs/polymorph/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/arthur/miniforge3/envs/polymorph/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/arthur/miniforge3/envs/polymorph/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/arthur/miniforge3/envs/polymorph/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/arthur/miniforge3/envs/polymorph/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not ha

✅ fcfp                 / LGBMClassifier  — bal_acc=0.698 (avant: 0.701, Δ=-0.003)  [229s, 20 iters]
Cache chargé [ecfp / csd_ml.csv] : (292373, 2048)


/home/arthur/miniforge3/envs/polymorph/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/arthur/miniforge3/envs/polymorph/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/arthur/miniforge3/envs/polymorph/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/arthur/miniforge3/envs/polymorph/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/arthur/miniforge3/envs/polymorph/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not ha

✅ ecfp                 / LGBMClassifier  — bal_acc=0.698 (avant: 0.701, Δ=-0.003)  [236s, 20 iters]
Cache chargé [desc2D / csd_ml.csv] : (292362, 223)


...

✅ fcfp-count           / LGBMClassifier  — bal_acc=0.706 (avant: 0.691, Δ=+0.016)  [258s, 20 iters]
Cache chargé [ecfp-count / csd_ml.csv] : (292373, 2048)


/home/arthur/miniforge3/envs/polymorph/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/arthur/miniforge3/envs/polymorph/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/arthur/miniforge3/envs/polymorph/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/arthur/miniforge3/envs/polymorph/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/arthur/miniforge3/envs/polymorph/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not ha

✅ ecfp-count           / LGBMClassifier  — bal_acc=0.706 (avant: 0.691, Δ=+0.016)  [239s, 20 iters]
Cache chargé [secfp / csd_ml.csv] : (292373, 2048)


/home/arthur/miniforge3/envs/polymorph/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/arthur/miniforge3/envs/polymorph/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/arthur/miniforge3/envs/polymorph/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/arthur/miniforge3/envs/polymorph/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/arthur/miniforge3/envs/polymorph/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not ha

✅ secfp                / LGBMClassifier  — bal_acc=0.698 (avant: 0.686, Δ=+0.012)  [253s, 20 iters]
⏭ secfp                / LogisticRegression — bal_acc=0.706 (avant tuning: 0.686)  (cache)
⏭ topological-count    / BalancedRF      — bal_acc=0.683 (avant tuning: 0.704)  (cache)
⏭ ecfp-count           / BalancedRF      — bal_acc=0.691 (avant tuning: 0.697)  (cache)
⏭ ecfp                 / BalancedRF      — bal_acc=0.689 (avant tuning: 0.696)  (cache)
⏭ fcfp                 / BalancedRF      — bal_acc=0.694 (avant tuning: 0.695)  (cache)
⏭ fcfp-count           / BalancedRF      — bal_acc=0.690 (avant tuning: 0.693)  (cache)
⏭ topological          / BalancedRF      — bal_acc=0.686 (avant tuning: 0.690)  (cache)
Cache chargé [ecfp / csd_ml.csv] : (292373, 2048)


In [ ]:
TUNING_RESULTS_CACHE  = "tuning_results.csv"
if os.path.exists(TUNING_RESULTS_CACHE):
    tuning_results = pd.read_csv(TUNING_RESULTS_CACHE).to_dict("records")
else:
    tuning_results = []

pd.set_option("display.max_rows", None)
tuning_df = pd.DataFrame(tuning_results).sort_values("balanced_accuracy", ascending=False).reset_index(drop=True)
tuning_df
